# HW3c on Colab

Runtime -> Change runtime type -> GPU, then run the cells top to bottom.

Checkpoints, YOLO runs, results and logs live in Google Drive (`MyDrive/245hw3c`), so if the session drops, reconnect and run everything again: finished parts are skipped by their own resume logic and unfinished ones continue from the last epoch.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE = '/content/drive/MyDrive/245hw3c'
REPO = '/content/245hw3c'
os.makedirs(DRIVE, exist_ok=True)

In [ ]:
# The repo is private: add a Colab secret named GITHUB_TOKEN (key icon on the left)
# holding a GitHub token that can read ericezhao/245hw3c. Not needed if the repo is public.
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    token = None
url = f'https://{token}@github.com/ericezhao/245hw3c.git' if token else 'https://github.com/ericezhao/245hw3c.git'

if not os.path.exists(REPO):
    !git clone -q {url} {REPO}
else:
    !git -C {REPO} pull -q
%cd {REPO}
!git log --oneline -1

In [ ]:
# Colab already ships a CUDA build of torch; this only adds what is missing.
!pip install -q -r requirements.txt
import torch
print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))

In [ ]:
# Point the output folders at Drive so they survive a disconnect.
import shutil
for name in ['checkpoints', 'yolo_runs', 'results', 'logs']:
    local, remote = os.path.join(REPO, name), os.path.join(DRIVE, name)
    os.makedirs(remote, exist_ok=True)
    if os.path.islink(local):
        continue
    if os.path.isdir(local):
        shutil.copytree(local, remote, dirs_exist_ok=True)
        shutil.rmtree(local)
    os.symlink(remote, local)
!ls -la {REPO} | grep -- '->'

In [ ]:
# First session downloads the data (~500 MB) and caches a tarball in Drive;
# later sessions just unpack it.
TAR = os.path.join(DRIVE, 'bdd_data.tar')
if os.path.exists(TAR):
    !tar -xf {TAR} -C {REPO}
else:
    !python prepare_bdd_mini.py
    !tar -cf {TAR} data/bdd100k_mini seg_masks
!ls data/bdd100k_mini/annotations

In [ ]:
!python -u part1_eda.py 2>&1 | tee logs/part1_eda.log

In [ ]:
!python -u part3_yolov8.py --batch 16 2>&1 | tee logs/part3_yolov8.log

In [ ]:
!python -u part2_faster_rcnn.py 2>&1 | tee logs/part2_faster_rcnn.log

In [ ]:
!python -u part4_deeplabv3.py 2>&1 | tee logs/part4_deeplabv3.log

In [ ]:
!python -u part5_mask_rcnn.py 2>&1 | tee logs/part5_mask_rcnn.log

In [ ]:
!python -u part6_analysis.py 2>&1 | tee logs/part6_analysis.log
!cp report.pdf {DRIVE}/
!cat results/metrics_table.csv

When everything is done, download `MyDrive/245hw3c/results`, `logs` and `report.pdf` (or share the Drive folder) and send them back so the results can be committed and the Word report written.